# Step 1 — Cleaning the ECB Survey of Professional Forecasters (SPF) Data

## Purpose

This notebook reads the raw ECB SPF individual-round CSV files, extracts the
**point forecasts** for **HICP inflation** and **real GDP growth**, and
produces a single clean panel dataset.

The final table has one row per *forecaster × survey round × target period*
and contains the columns we need for the later modelling steps:

| column | meaning |
|---|---|
| `survey_round` | the quarter the survey was conducted (e.g. `2020Q1`) |
| `variable` | `HICP` (inflation) or `RGDP` (real GDP growth) |
| `target_period` | the period the forecast refers to (e.g. `2020`, `2021Dec`, `2020Q3`) |
| `fct_source` | anonymous forecaster ID (integer, stable across rounds) |
| `point_forecast` | the forecaster's point estimate |
| `horizon_type` | label for the forecast horizon: `current_year`, `next_year`, `year_after_next`, `longer_term`, `rolling_1y`, `rolling_2y` |

We keep **all** available calendar-year and rolling horizons at this stage.
Downstream notebooks will filter to the rolling one-year-ahead horizon.

---
## 1 — Imports and setup

In [1]:
import os
import glob
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

pandas  3.0.2
numpy   2.4.4


In [2]:
# Path to the folder that contains one CSV per survey round (1999Q1.csv … 2026Q2.csv)
DATA_DIR = os.path.join("Data", "SPF_individual_forecasts")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"Found {len(csv_files)} survey-round files.")
print(f"First : {os.path.basename(csv_files[0])}")
print(f"Last  : {os.path.basename(csv_files[-1])}")

Found 110 survey-round files.
First : 1999Q1.csv
Last  : 2026Q2.csv


---
## 2 — Understanding the raw CSV layout

Each CSV file corresponds to **one survey round** (e.g. `2020Q1.csv`).
Inside a single file, several tables are stacked vertically, separated by
blank rows:

1. **HICP inflation** — header row starts with
   `INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN HICP`
2. **Core inflation (HICPX)** — header starts with
   `CORE INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN CORE`
3. **Real GDP growth** — header starts with
   `GROWTH EXPECTATIONS; YEAR-ON-YEAR CHANGE IN REAL GDP`
4. **Unemployment** — header starts with
   `EXPECTED UNEMPLOYMENT RATE`

Below the four forecast tables there may also be an assumptions block
(ECB interest rate, oil prices, exchange rates, labour costs).

### Column structure inside each table

| column | content |
|---|---|
| `TARGET_PERIOD` | The period being forecast. Format depends on the variable and horizon: a plain year like `2020` for calendar-year forecasts, a month like `2020Dec` for HICP rolling horizons, or a quarter like `2020Q3` for GDP rolling horizons. |
| `FCT_SOURCE` | Integer ID of the individual forecaster (anonymous, but stable over time). |
| `POINT` | The forecaster's point estimate (single number). |
| remaining columns | Probability bins for the density forecast (we ignore these). |

We only need `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.

### Forecast horizons

The ECB asks forecasters for **up to six horizons** (since 2001 Q2):

1. **Current calendar year** — e.g. in 2020Q1 the target period `2020`.
2. **Next calendar year** — e.g. `2021`.
3. **Calendar year after next** — e.g. `2022`.
4. **Longer-term** — four or five calendar years ahead.
5. **Rolling 1-year-ahead** — for HICP this is a month
   (e.g. `2020Dec`), for GDP a quarter (e.g. `2020Q3`). The idea is
   that the target is always approximately 12 months from the last
   available data release.
6. **Rolling 2-year-ahead** — same concept, two years out
   (e.g. `2021Dec` / `2021Q3`).

The rolling horizons are the most interesting for our project because
they give a **fixed forecasting distance**, making comparisons across
time cleaner.

Let's peek at a raw file to verify our understanding:

In [3]:
# Quick peek at the first 10 lines of 2020Q1.csv
sample_path = os.path.join(DATA_DIR, "2020Q1.csv")
with open(sample_path, "r") as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"line {i+1:>4}: {line.rstrip()[:120]}")

line    1: INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN HICP,,,,,,,,,,,,,,,,,,,,,,,
line    2: TARGET_PERIOD,FCT_SOURCE,POINT,TN1_0,FN1_0TN0_6,FN0_5TN0_1,F0_0T0_4,F0_5T0_9,F1_0T1_4,F1_5T1_9,F2_0T2_4,F2_5T2_9,F3_0T3_
line    3: 2020,1,1.2,,1,2,4,15,44,22,8,3,1,,,,,,,,,,,
line    4: 2020,2,1.4,0,0,0,0,0,50,50,0,0,0,0,0,,,,,,,,,
line    5: 2020,4,1.2,,,,10,20,40,20,10,,,,,,,,,,,,,
line    6: 2020,5,1,,,,7.5,25,40,20,7.5,,,,,,,,,,,,,
line    7: 2020,6,1.2,,,,,10,50,25,10,5,,,,,,,,,,,,
line    8: 2020,7,1.1766426795,,,,,,,,,,,,,,,,,,,,,
line    9: 2020,8,1,,,,5,30,35,20,10,,,,,,,,,,,,,
line   10: 2020,10,1,,,,5,15,70,10,,,,,,,,,,,,,,


Line 1 is the section header for HICP. Line 2 is the column header row.
Lines 3 onwards contain the actual data. Note how `TARGET_PERIOD` values
like `2020` (calendar year) and `2020Dec` (rolling 1-year-ahead) are
intermixed — the section header tells us the *variable* (HICP vs RGDP),
and the target period format tells us the *horizon type*.

---
## 3 — Parsing a single CSV file

Our strategy for parsing one file:

1. Read the file line by line.
2. Detect section headers to know which variable (HICP / RGDP) we are in.
3. When we see a `TARGET_PERIOD` header row, read subsequent data rows
   until we hit a blank line or a new section header.
4. Keep only `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.
5. Tag every row with the current variable name.

We wrap this in a function so we can apply it to all 110 files.

In [4]:
# -------------------------------------------------------------------
# Section-header patterns that tell us which variable block we are in.
# We only care about HICP and RGDP; the function skips everything else.
# -------------------------------------------------------------------
SECTION_PATTERNS = {
    "HICP": re.compile(r"^INFLATION EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN HICP"),
    "RGDP": re.compile(r"^GROWTH EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN REAL GDP"),
}

# Patterns that signal the START of a section we want to skip.
# When we see one of these while inside an HICP/RGDP block we stop reading.
SKIP_PATTERNS = [
    re.compile(r"^CORE INFLATION"),
    re.compile(r"^EXPECTED UNEMPLOYMENT"),
    re.compile(r"^ASSUMPTIONS"),
]


def parse_spf_file(filepath: str) -> pd.DataFrame:
    """
    Parse one SPF survey-round CSV file and return a DataFrame with columns:
        TARGET_PERIOD, FCT_SOURCE, POINT, variable

    Only HICP and RGDP blocks are extracted; all other blocks are skipped.
    """
    rows = []  # accumulator for (target_period, fct_source, point, variable)
    current_var = None  # which variable block are we inside?

    with open(filepath, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            # --- Check for section headers we want ---
            matched_var = None
            for var_name, pattern in SECTION_PATTERNS.items():
                if pattern.search(line):
                    matched_var = var_name
                    break

            if matched_var:
                current_var = matched_var
                continue  # move to the next line (column header follows)

            # --- Check for section headers we want to skip ---
            if any(p.search(line) for p in SKIP_PATTERNS):
                current_var = None  # we are no longer in an HICP/RGDP block
                continue

            # --- Skip column-header rows and blank lines ---
            if line.startswith("TARGET_PERIOD") or line == "" or line.replace(",", "") == "":
                continue

            # --- If we are inside an HICP or RGDP block, parse the data row ---
            if current_var is not None:
                fields = line.split(",")
                target_period = fields[0].strip()
                fct_source = fields[1].strip()
                point_raw = fields[2].strip()

                # Skip rows where POINT is missing (some forecasters only
                # provide density forecasts without a point estimate)
                if point_raw == "" or target_period == "" or fct_source == "":
                    continue

                rows.append((target_period, fct_source, point_raw, current_var))

    df = pd.DataFrame(rows, columns=["TARGET_PERIOD", "FCT_SOURCE", "POINT", "variable"])
    return df


# Quick test on 2020Q1
test_df = parse_spf_file(os.path.join(DATA_DIR, "2020Q1.csv"))
print(f"Rows parsed from 2020Q1.csv: {len(test_df)}")
print(f"Variables found: {test_df['variable'].unique()}")
print(f"Target periods:  {sorted(test_df['TARGET_PERIOD'].unique())}")
test_df.head(8)

Rows parsed from 2020Q1.csv: 723
Variables found: <StringArray>
['HICP', 'RGDP']
Length: 2, dtype: str
Target periods:  ['2020', '2020Dec', '2020Q3', '2021', '2021Dec', '2021Q3', '2022', '2024']


,TARGET_PERIOD,FCT_SOURCE,POINT,variable
0,2020,1,1.2,HICP
1,2020,2,1.4,HICP
2,2020,4,1.2,HICP
3,2020,5,1,HICP
4,2020,6,1.2,HICP
5,2020,7,1.1766426795,HICP
6,2020,8,1,HICP
7,2020,10,1,HICP


The parser correctly picks up only the HICP and RGDP sections.
For HICP the rolling horizons are months (e.g. `2020Dec`, `2021Dec`)
and for RGDP they are quarters (e.g. `2020Q3`, `2021Q3`).

---
## 4 — Parsing all 110 survey rounds

We loop over every CSV in the `Data/SPF_individual_forecasts` folder.
The **survey round** (e.g. `2020Q1`) is extracted from the filename.

In [5]:
frames = []

for fpath in csv_files:
    # Extract the survey round from the filename: "2020Q1.csv" → "2020Q1"
    survey_round = os.path.splitext(os.path.basename(fpath))[0]

    df = parse_spf_file(fpath)
    df["survey_round"] = survey_round  # tag every row
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f"Total rows across all rounds: {len(raw):,}")
print(f"Survey rounds: {raw['survey_round'].nunique()}")
raw.head()

Total rows across all rounds: 61,677


Survey rounds: 110


,TARGET_PERIOD,FCT_SOURCE,POINT,variable,survey_round
0,1999,1,1,HICP,1999Q1
1,1999,2,1,HICP,1999Q1
2,1999,3,.8,HICP,1999Q1
3,1999,4,1.2,HICP,1999Q1
4,1999,5,.8,HICP,1999Q1


---
## 5 — Type conversions and basic cleaning

The values we parsed are still strings. We need to:

1. Convert `FCT_SOURCE` to integer (forecaster IDs are integers).
2. Convert `POINT` to float (the point forecast).
3. Drop any row where the conversion fails — this catches the rare
   cases where a field is garbled or empty.

In [6]:
# Convert POINT to numeric — non-numeric strings become NaN
raw["POINT"] = pd.to_numeric(raw["POINT"], errors="coerce")

# Convert FCT_SOURCE to integer
raw["FCT_SOURCE"] = pd.to_numeric(raw["FCT_SOURCE"], errors="coerce")

# Count how many rows have a missing point forecast after conversion
n_missing_point = raw["POINT"].isna().sum()
n_missing_fct = raw["FCT_SOURCE"].isna().sum()
print(f"Missing POINT values after conversion: {n_missing_point}")
print(f"Missing FCT_SOURCE after conversion:   {n_missing_fct}")

# Drop rows with missing point forecasts
raw = raw.dropna(subset=["POINT", "FCT_SOURCE"]).copy()
raw["FCT_SOURCE"] = raw["FCT_SOURCE"].astype(int)

print(f"\nRows remaining: {len(raw):,}")
raw.dtypes

Missing POINT values after conversion: 0
Missing FCT_SOURCE after conversion:   0

Rows remaining: 61,677


TARGET_PERIOD        str
FCT_SOURCE         int64
POINT            float64
variable             str
survey_round         str
dtype: object

---
## 6 — Classifying the forecast horizon

This is the most important cleaning step. Each row has a `TARGET_PERIOD`
and a `survey_round`. From these two we can figure out **which forecast
horizon** the row belongs to.

### Horizon classification rules

| horizon label | how to detect it |
|---|---|
| `rolling_1y` | HICP/RGDP target has a month or quarter suffix. Within each `survey_round × variable`, it is the **earliest** rolling target period. |
| `rolling_2y` | Same month/quarter format, but it is the **second** rolling target period within that `survey_round × variable`. |
| `current_year` | Target is a plain year equal to the survey year. |
| `next_year` | Target is a plain year equal to survey year + 1. |
| `year_after_next` | Target is a plain year equal to survey year + 2. |
| `longer_term` | Target is a plain year ≥ survey year + 3. |

The trick is that **rolling horizons always contain a month or quarter
suffix** (like `Dec` or `Q3`), whereas calendar-year horizons are just
a four-digit year. We do **not** classify rolling horizons only from
`target_year - survey_year`, because early rounds can have both `1999Dec`
and `2000Dec` in the same survey. The ECB description says these are the
one-year- and two-year-ahead rolling horizons, respectively.

We implement this classification with a function:

In [7]:
# Regex patterns for target-period formats
RE_YEAR_ONLY = re.compile(r"^(\d{4})$")                     # e.g. "2020"
RE_MONTH = re.compile(r"^(\d{4})(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$")  # e.g. "2020Dec"
RE_QUARTER = re.compile(r"^(\d{4})Q([1-4])$")               # e.g. "2020Q3"


def target_period_sort_key(target_period: str) -> pd.Timestamp:
    """Convert month/quarter TARGET_PERIOD strings to sortable dates."""
    m_month = RE_MONTH.match(target_period)
    if m_month:
        year, month_name = m_month.groups()
        month_num = pd.to_datetime(month_name, format="%b").month
        return pd.Timestamp(int(year), month_num, 1)

    m_quarter = RE_QUARTER.match(target_period)
    if m_quarter:
        year, quarter = m_quarter.groups()
        month_num = (int(quarter) - 1) * 3 + 1
        return pd.Timestamp(int(year), month_num, 1)

    return pd.NaT


def classify_calendar_horizon(target_period: str, survey_round: str) -> str:
    """
    Classify plain calendar-year targets. Rolling month/quarter targets
    are assigned below using their within-round ordering.
    """
    survey_year = int(survey_round[:4])

    if RE_MONTH.match(target_period) or RE_QUARTER.match(target_period):
        return "rolling_unassigned"

    # --- Calendar-year horizons (plain four-digit year) ---
    m_year = RE_YEAR_ONLY.match(target_period)
    if m_year:
        target_year = int(m_year.group(1))
        diff = target_year - survey_year
        if diff == 0:
            return "current_year"
        elif diff == 1:
            return "next_year"
        elif diff == 2:
            return "year_after_next"
        else:
            return "longer_term"

    return "unknown"


# First classify calendar-year rows and mark rolling rows for group-wise assignment.
raw["horizon_type"] = raw.apply(
    lambda r: classify_calendar_horizon(r["TARGET_PERIOD"], r["survey_round"]),
    axis=1,
)

# The ECB SPF defines rolling horizons as one and two years ahead of the
# latest available observation. In the raw files, those are the first and
# second month/quarter target periods within each survey round and variable.
rolling_mask = raw["horizon_type"] == "rolling_unassigned"
rolling_targets = (
    raw.loc[rolling_mask, ["survey_round", "variable", "TARGET_PERIOD"]]
    .drop_duplicates()
    .assign(target_sort=lambda d: d["TARGET_PERIOD"].apply(target_period_sort_key))
    .sort_values(["survey_round", "variable", "target_sort"])
)
rolling_targets["rolling_rank"] = (
    rolling_targets.groupby(["survey_round", "variable"]).cumcount() + 1
)
rolling_targets["rolling_label"] = rolling_targets["rolling_rank"].map(
    {1: "rolling_1y", 2: "rolling_2y"}
).fillna("rolling_longer")

raw = raw.merge(
    rolling_targets[["survey_round", "variable", "TARGET_PERIOD", "rolling_label"]],
    on=["survey_round", "variable", "TARGET_PERIOD"],
    how="left",
)
raw.loc[rolling_mask, "horizon_type"] = raw.loc[rolling_mask, "rolling_label"]
raw = raw.drop(columns=["rolling_label"])

print("Horizon type distribution:")
print(raw["horizon_type"].value_counts().sort_index())

Horizon type distribution:
horizon_type
current_year       12490
longer_term         9115
next_year          12236
rolling_1y         10726
rolling_2y          9511
rolling_longer       253
year_after_next     7346
Name: count, dtype: int64


In [8]:
# Sanity check: there should be zero "unknown" horizons
n_unknown = (raw["horizon_type"] == "unknown").sum()
if n_unknown > 0:
    print(f"WARNING: {n_unknown} rows with unknown horizon type!")
    print(raw.loc[raw["horizon_type"] == "unknown", "TARGET_PERIOD"].unique())
else:
    print("All rows classified successfully — no unknowns.")

All rows classified successfully — no unknowns.


---
## 7 — Rename columns to final schema

We rename the raw column names to the clean, lowercase names that we
will use throughout the project.

In [9]:
clean = raw.rename(columns={
    "TARGET_PERIOD": "target_period",
    "FCT_SOURCE":    "fct_source",
    "POINT":         "point_forecast",
})

# Reorder columns into a logical sequence
clean = clean[["survey_round", "variable", "target_period",
               "fct_source", "point_forecast", "horizon_type"]]

clean.head(10)

,survey_round,variable,target_period,fct_source,point_forecast,horizon_type
0,1999Q1,HICP,1999,1,1.0,current_year
1,1999Q1,HICP,1999,2,1.0,current_year
2,1999Q1,HICP,1999,3,0.8,current_year
3,1999Q1,HICP,1999,4,1.2,current_year
4,1999Q1,HICP,1999,5,0.8,current_year
5,1999Q1,HICP,1999,6,0.9,current_year
6,1999Q1,HICP,1999,7,0.8,current_year
7,1999Q1,HICP,1999,9,0.7,current_year
8,1999Q1,HICP,1999,10,1.2,current_year
9,1999Q1,HICP,1999,11,1.2,current_year


---
## 8 — Extract the rolling and calendar-year forecast columns

For the wide modelling table, we keep the main rolling horizon and also
preserve the additional rolling horizons when they are available:

- **`rolling_1y_forecast`** — the point forecast for the rolling
  one-year-ahead horizon.
- **`rolling_2y_forecast`** — the point forecast for the rolling
  two-year-ahead horizon.
- **`rolling_longer_forecast`** — the longer rolling horizon, available
  only in some early Q1 survey rounds.
- **`next_year_forecast`** — the point forecast for the next calendar year.

We pivot the data so that each forecaster × survey round has these
these numbers side by side. This is useful because we can later compare
forecast horizons and check consistency.

### Why keep both?

The **rolling 1-year-ahead** horizon is the main one for the neural
network project (it gives a fixed forecast distance). The **rolling 2-year**
and **rolling longer** horizons are useful for robustness checks or extra
features. The **next calendar year** horizon is the most commonly reported
in the SPF and is useful as an additional feature or cross-check.

In [10]:
def make_horizon_slice(horizon_type, target_col, forecast_col):
    """Return one horizon as a wide-table slice."""
    return (
        clean
        .loc[clean["horizon_type"] == horizon_type]
        .rename(columns={"point_forecast": forecast_col,
                         "target_period":  target_col})
        [["survey_round", "variable", "fct_source", target_col, forecast_col]]
    )


# --- Rolling horizon forecasts ---
rolling_1y = make_horizon_slice("rolling_1y", "rolling_1y_target", "rolling_1y_forecast")
rolling_2y = make_horizon_slice("rolling_2y", "rolling_2y_target", "rolling_2y_forecast")
rolling_longer = make_horizon_slice(
    "rolling_longer", "rolling_longer_target", "rolling_longer_forecast"
)

# --- Next-year calendar-year forecasts ---
next_year = make_horizon_slice("next_year", "next_year_target", "next_year_forecast")

print(f"Rolling 1y rows : {len(rolling_1y):,}")
print(f"Rolling 2y rows : {len(rolling_2y):,}")
print(f"Rolling longer  : {len(rolling_longer):,}")
print(f"Next year rows  : {len(next_year):,}")

Rolling 1y rows : 10,726
Rolling 2y rows : 9,511
Rolling longer  : 253
Next year rows  : 12,236


In [11]:
# Merge the horizon slices together.
# We use outer merges so we keep forecasters who reported some horizons but not others.
horizon_frames = [rolling_1y, rolling_2y, rolling_longer, next_year]
merged = horizon_frames[0].copy()

for frame in horizon_frames[1:]:
    merged = pd.merge(
        merged,
        frame,
        on=["survey_round", "variable", "fct_source"],
        how="outer",
    )

print(f"Merged rows: {len(merged):,}")
print(f"\nMissing rolling_1y_forecast: {merged['rolling_1y_forecast'].isna().sum():,}")
print(f"Missing rolling_2y_forecast: {merged['rolling_2y_forecast'].isna().sum():,}")
print(f"Missing rolling_longer_forecast: {merged['rolling_longer_forecast'].isna().sum():,}")
print(f"Missing next_year_forecast:  {merged['next_year_forecast'].isna().sum():,}")
merged.head(10)

Merged rows: 12,358

Missing rolling_1y_forecast: 1,632
Missing rolling_2y_forecast: 2,847
Missing rolling_longer_forecast: 12,105
Missing next_year_forecast:  122


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast
0,1999Q1,HICP,1,1999Dec,1.2,2000Dec,1.7,2003Dec,1.8,2000,1.5
1,1999Q1,HICP,2,1999Dec,1.2,2000Dec,1.2,2003Dec,1.6,2000,1.2
2,1999Q1,HICP,3,1999Dec,0.8,2000Dec,1.2,2003Dec,1.7,2000,1.0
3,1999Q1,HICP,4,1999Dec,1.3,2000Dec,1.8,2003Dec,2.0,2000,1.6
4,1999Q1,HICP,5,1999Dec,1.1,2000Dec,1.5,2003Dec,1.7,2000,1.4
5,1999Q1,HICP,6,1999Dec,1.1,2000Dec,1.3,2003Dec,1.9,2000,1.3
6,1999Q1,HICP,7,1999Dec,0.8,2000Dec,0.8,NaN,NaN,2000,0.8
7,1999Q1,HICP,9,1999Dec,0.6,2000Dec,0.7,2003Dec,1.3,2000,0.6
8,1999Q1,HICP,10,1999Dec,1.1,2000Dec,1.6,2003Dec,1.6,2000,1.3
9,1999Q1,HICP,11,1999Dec,1.5,2000Dec,1.6,2003Dec,2.0,2000,1.5


---
## 9 — Add current-year forecast as an extra feature

The current-year forecast is another useful piece of information.
By the time Q3 or Q4 rolls around, the current-year forecast is nearly
a realized value, so it captures how much information about the
macro economy the forecaster already has. We add it as an additional
column.

In [12]:
current_year = (
    clean
    .loc[clean["horizon_type"] == "current_year"]
    .rename(columns={"point_forecast": "current_year_forecast",
                     "target_period":  "current_year_target"})
    [["survey_round", "variable", "fct_source",
      "current_year_target", "current_year_forecast"]]
)

merged = pd.merge(
    merged,
    current_year,
    on=["survey_round", "variable", "fct_source"],
    how="left",
)

print(f"After adding current-year forecast: {len(merged):,} rows")
merged.head(10)

After adding current-year forecast: 12,358 rows


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast
0,1999Q1,HICP,1,1999Dec,1.2,2000Dec,1.7,2003Dec,1.8,2000,1.5,1999,1.0
1,1999Q1,HICP,2,1999Dec,1.2,2000Dec,1.2,2003Dec,1.6,2000,1.2,1999,1.0
2,1999Q1,HICP,3,1999Dec,0.8,2000Dec,1.2,2003Dec,1.7,2000,1.0,1999,0.8
3,1999Q1,HICP,4,1999Dec,1.3,2000Dec,1.8,2003Dec,2.0,2000,1.6,1999,1.2
4,1999Q1,HICP,5,1999Dec,1.1,2000Dec,1.5,2003Dec,1.7,2000,1.4,1999,0.8
5,1999Q1,HICP,6,1999Dec,1.1,2000Dec,1.3,2003Dec,1.9,2000,1.3,1999,0.9
6,1999Q1,HICP,7,1999Dec,0.8,2000Dec,0.8,NaN,NaN,2000,0.8,1999,0.8
7,1999Q1,HICP,9,1999Dec,0.6,2000Dec,0.7,2003Dec,1.3,2000,0.6,1999,0.7
8,1999Q1,HICP,10,1999Dec,1.1,2000Dec,1.6,2003Dec,1.6,2000,1.3,1999,1.2
9,1999Q1,HICP,11,1999Dec,1.5,2000Dec,1.6,2003Dec,2.0,2000,1.5,1999,1.2


---
## 10 — Parse the survey round into year and quarter columns

Having `survey_year` and `survey_quarter` as separate numeric columns
will be handy for sorting, plotting, and later for creating time-based
features like "survey quarter" dummies.

In [13]:
merged["survey_year"] = merged["survey_round"].str[:4].astype(int)
merged["survey_quarter"] = merged["survey_round"].str[-1].astype(int)

merged.head()

,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
0,1999Q1,HICP,1,1999Dec,1.2,2000Dec,1.7,2003Dec,1.8,2000,1.5,1999,1.0,1999,1
1,1999Q1,HICP,2,1999Dec,1.2,2000Dec,1.2,2003Dec,1.6,2000,1.2,1999,1.0,1999,1
2,1999Q1,HICP,3,1999Dec,0.8,2000Dec,1.2,2003Dec,1.7,2000,1.0,1999,0.8,1999,1
3,1999Q1,HICP,4,1999Dec,1.3,2000Dec,1.8,2003Dec,2.0,2000,1.6,1999,1.2,1999,1
4,1999Q1,HICP,5,1999Dec,1.1,2000Dec,1.5,2003Dec,1.7,2000,1.4,1999,0.8,1999,1


---
## 11 — Summary statistics and quality checks

In [14]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total rows             : {len(merged):,}")
print(f"Unique survey rounds   : {merged['survey_round'].nunique()}")
print(f"Unique forecasters     : {merged['fct_source'].nunique()}")
print(f"Variables              : {sorted(merged['variable'].unique())}")
print(f"Year range             : {merged['survey_year'].min()} – {merged['survey_year'].max()}")
print()

for var in ["HICP", "RGDP"]:
    sub = merged[merged["variable"] == var]
    print(f"--- {var} ---")
    print(f"  Rows                   : {len(sub):,}")
    print(f"  Survey rounds          : {sub['survey_round'].nunique()}")
    print(f"  Forecasters            : {sub['fct_source'].nunique()}")
    print(f"  rolling_1y available    : {sub['rolling_1y_forecast'].notna().sum():,}")
    print(f"  rolling_2y available    : {sub['rolling_2y_forecast'].notna().sum():,}")
    print(f"  rolling_longer available: {sub['rolling_longer_forecast'].notna().sum():,}")
    print(f"  next_year available     : {sub['next_year_forecast'].notna().sum():,}")
    print(f"  current_year available  : {sub['current_year_forecast'].notna().sum():,}")
    print()

DATASET OVERVIEW
Total rows             : 12,358
Unique survey rounds   : 110
Unique forecasters     : 113
Variables              : ['HICP', 'RGDP']
Year range             : 1999 – 2026

--- HICP ---
  Rows                   : 6,158
  Survey rounds          : 110
  Forecasters            : 113
  rolling_1y available    : 5,355
  rolling_2y available    : 4,708
  rolling_longer available: 130
  next_year available     : 6,103
  current_year available  : 6,155

--- RGDP ---
  Rows                   : 6,200
  Survey rounds          : 110
  Forecasters            : 113
  rolling_1y available    : 5,371
  rolling_2y available    : 4,803
  rolling_longer available: 123
  next_year available     : 6,133
  current_year available  : 6,198



In [15]:
# Descriptive statistics for the point forecasts
forecast_cols = [
    "rolling_1y_forecast",
    "rolling_2y_forecast",
    "rolling_longer_forecast",
    "next_year_forecast",
    "current_year_forecast",
]

merged[forecast_cols].describe().round(2)

,rolling_1y_forecast,rolling_2y_forecast,rolling_longer_forecast,next_year_forecast,current_year_forecast
count,10726.00,9511.00,253.00,12236.00,12353.00
mean,1.66,1.82,2.19,1.82,1.59
std,1.26,0.57,0.49,0.89,1.79
min,-10.95,-1.11,0.25,-2.01,-13.00
25%,1.20,1.50,1.80,1.40,1.00
50%,1.60,1.80,2.25,1.70,1.70
75%,2.00,2.01,2.50,2.10,2.20
max,16.60,12.50,3.30,14.50,9.30


In [16]:
# Check for obvious outliers: point forecasts outside a plausible range
# HICP and RGDP in the euro area have historically stayed within roughly -15% to +15%
for var in ["HICP", "RGDP"]:
    sub = merged[merged["variable"] == var]
    for col in forecast_cols:
        vals = sub[col].dropna()
        extreme = vals[(vals < -15) | (vals > 15)]
        if len(extreme) > 0:
            print(f"  {var} / {col}: {len(extreme)} values outside [-15, 15]")
        else:
            print(f"  {var} / {col}: all values in plausible range")

  HICP / rolling_1y_forecast: all values in plausible range
  HICP / rolling_2y_forecast: all values in plausible range
  HICP / rolling_longer_forecast: all values in plausible range
  HICP / next_year_forecast: all values in plausible range
  HICP / current_year_forecast: all values in plausible range
  RGDP / rolling_1y_forecast: 1 values outside [-15, 15]
  RGDP / rolling_2y_forecast: all values in plausible range
  RGDP / rolling_longer_forecast: all values in plausible range
  RGDP / next_year_forecast: all values in plausible range
  RGDP / current_year_forecast: all values in plausible range


---
## 12 — Inspect sample rows to make sure everything looks right

In [17]:
# Show a few rows for HICP, 2020Q1
print("HICP forecasts in 2020Q1 (first 8 forecasters):")
merged.loc[
    (merged["survey_round"] == "2020Q1") & (merged["variable"] == "HICP")
].head(8)

HICP forecasts in 2020Q1 (first 8 forecasters):


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
9360,2020Q1,HICP,1,2020Dec,1.2,2021Dec,1.5,NaN,NaN,2021,1.5,2020,1.2,2020,1
9361,2020Q1,HICP,2,2020Dec,1.4,2021Dec,1.5,NaN,NaN,2021,1.5,2020,1.4,2020,1
9362,2020Q1,HICP,4,2020Dec,1.2,2021Dec,1.2,NaN,NaN,2021,1.2,2020,1.2,2020,1
9363,2020Q1,HICP,5,2020Dec,1.0,2021Dec,1.5,NaN,NaN,2021,1.5,2020,1.0,2020,1
9364,2020Q1,HICP,6,2020Dec,1.1,2021Dec,1.2,NaN,NaN,2021,1.1,2020,1.2,2020,1
9365,2020Q1,HICP,8,2020Dec,0.9,2021Dec,1.3,NaN,NaN,2021,1.1,2020,1.0,2020,1
9366,2020Q1,HICP,10,2020Dec,0.9,2021Dec,1.1,NaN,NaN,2021,1.0,2020,1.0,2020,1
9367,2020Q1,HICP,14,2020Dec,1.1,2021Dec,1.7,NaN,NaN,2021,1.7,2020,1.1,2020,1


In [18]:
# Show a few rows for RGDP, 2020Q1
print("RGDP forecasts in 2020Q1 (first 8 forecasters):")
merged.loc[
    (merged["survey_round"] == "2020Q1") & (merged["variable"] == "RGDP")
].head(8)

RGDP forecasts in 2020Q1 (first 8 forecasters):


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
9427,2020Q1,RGDP,1,2020Q3,1.000000,2021Q3,1.3,NaN,NaN,2021,1.3,2020,0.900000,2020,1
9428,2020Q1,RGDP,2,2020Q3,1.400000,2021Q3,1.7,NaN,NaN,2021,1.7,2020,1.400000,2020,1
9429,2020Q1,RGDP,4,2020Q3,0.600000,2021Q3,1.2,NaN,NaN,2021,1.1,2020,0.600000,2020,1
9430,2020Q1,RGDP,5,2020Q3,1.000000,2021Q3,1.5,NaN,NaN,2021,1.5,2020,1.000000,2020,1
9431,2020Q1,RGDP,6,2020Q3,0.800000,2021Q3,1.0,NaN,NaN,2021,0.9,2020,0.800000,2020,1
9432,2020Q1,RGDP,7,2020Q3,1.200056,NaN,NaN,NaN,NaN,NaN,NaN,2020,1.115976,2020,1
9433,2020Q1,RGDP,8,2020Q3,0.800000,2021Q3,1.0,NaN,NaN,2021,1.0,2020,0.800000,2020,1
9434,2020Q1,RGDP,10,2020Q3,0.800000,2021Q3,1.4,NaN,NaN,2021,1.3,2020,0.800000,2020,1


---
## 13 — Also keep the long-form (all horizons) dataset

The `merged` table above is the **wide** format with rolling_1y,
rolling_2y, rolling_longer, next_year, and current_year as separate
columns. But it's also useful to keep the **long** format (`clean`) that
has every single horizon as a separate row, in case we need to inspect
year-after-next or longer-term calendar-year forecasts later.

We save both.

In [19]:
# Add survey_year and survey_quarter to the long-form dataset too
clean["survey_year"] = clean["survey_round"].str[:4].astype(int)
clean["survey_quarter"] = clean["survey_round"].str[-1].astype(int)

print("Long-form dataset shape:", clean.shape)
print("Wide-form dataset shape:", merged.shape)

Long-form dataset shape: (61677, 8)
Wide-form dataset shape: (12358, 15)


---
## 14 — Save the cleaned datasets to CSV

In [20]:
# Create output directory if it doesn't exist
os.makedirs("Data", exist_ok=True)

# Long form: one row per forecaster × round × target period × variable
clean.to_csv(os.path.join("Data", "spf_clean_long.csv"), index=False)
print(f"Saved long-form dataset : Data/spf_clean_long.csv  ({len(clean):,} rows)")

# Wide form: one row per forecaster × round × variable, with selected horizons as columns
merged.to_csv(os.path.join("Data", "spf_clean_wide.csv"), index=False)
print(f"Saved wide-form dataset : Data/spf_clean_wide.csv  ({len(merged):,} rows)")

Saved long-form dataset : Data/spf_clean_long.csv  (61,677 rows)
Saved wide-form dataset : Data/spf_clean_wide.csv  (12,358 rows)


---
## 15 — Final schema of the wide dataset

This is the dataset we will use going forward. Here is every column
and what it means:

In [21]:
schema_info = {
    "column": merged.columns.tolist(),
    "dtype": [str(merged[c].dtype) for c in merged.columns],
    "non_null": [merged[c].notna().sum() for c in merged.columns],
    "example": [merged[c].dropna().iloc[0] if merged[c].notna().any() else None for c in merged.columns],
}
pd.DataFrame(schema_info)

,column,dtype,non_null,example
0,survey_round,str,12358,1999Q1
1,variable,str,12358,HICP
2,fct_source,int64,12358,1
3,rolling_1y_target,str,10726,1999Dec
4,rolling_1y_forecast,float64,10726,1.2
5,rolling_2y_target,str,9511,2000Dec
6,rolling_2y_forecast,float64,9511,1.7
7,rolling_longer_target,str,253,2003Dec
8,rolling_longer_forecast,float64,253,1.8
9,next_year_target,str,12236,2000


### Column descriptions

| column | description |
|---|---|
| `survey_round` | Quarter when the ECB conducted the survey (e.g. `2020Q1`). Extracted from the filename. |
| `variable` | Which macroeconomic variable: `HICP` = year-on-year change in the Harmonised Index of Consumer Prices (inflation); `RGDP` = year-on-year change in real GDP (growth). |
| `fct_source` | Anonymous integer ID for each forecaster. The same number refers to the same institution across all rounds. |
| `rolling_1y_target` | The specific target period for the rolling one-year-ahead forecast (e.g. `2020Dec` for HICP, `2020Q3` for RGDP). |
| `rolling_1y_forecast` | The forecaster's point estimate for the rolling one-year-ahead horizon. This is our **main forecast** column. |
| `rolling_2y_target` | The specific target period for the rolling two-year-ahead forecast (e.g. `2000Dec` in the 1999Q1 HICP survey). |
| `rolling_2y_forecast` | The forecaster's point estimate for the rolling two-year-ahead horizon. |
| `rolling_longer_target` | The specific target period for the longer rolling horizon, when it exists (e.g. `2003Dec` in the 1999Q1 HICP survey). |
| `rolling_longer_forecast` | The forecaster's point estimate for the longer rolling horizon. This is sparse because it is only available in some early rounds. |
| `next_year_target` | The calendar year being forecast as "next year" (e.g. `2021` when surveyed in 2020). |
| `next_year_forecast` | The forecaster's point estimate for the next calendar year. |
| `current_year_target` | The calendar year being forecast as "current year". |
| `current_year_forecast` | The forecaster's point estimate for the current calendar year. |
| `survey_year` | Numeric year of the survey (integer). |
| `survey_quarter` | Numeric quarter of the survey (1–4). |

---
## Summary

We started with 110 raw CSV files from the ECB SPF, each containing
multiple stacked tables for different macro variables.

**What we did:**

1. Wrote a parser that reads each CSV, identifies the HICP and RGDP
   sections by their header text, and extracts `TARGET_PERIOD`,
   `FCT_SOURCE`, and `POINT`.
2. Tagged every row with the `survey_round` from the filename.
3. Converted types (string → numeric) and dropped rows with missing
   point forecasts.
4. Classified every row into a horizon type (`rolling_1y`, `next_year`,
   `current_year`, etc.). Calendar-year horizons use the year difference;
   rolling horizons use the within-round ordering described in the ECB SPF
   dataset documentation.
5. Pivoted the selected horizons into separate columns:
   `rolling_1y_forecast`, `rolling_2y_forecast`,
   `rolling_longer_forecast`, `next_year_forecast`, and
   `current_year_forecast`.
6. Added `survey_year` and `survey_quarter` for convenience.
7. Saved both a long-form and a wide-form CSV.

**What we have now:**

- A clean panel dataset at the *forecaster × survey round × variable*
  level.
- Point forecasts for three horizons, ready for the next step:
  merging with realized outcomes and computing forecast errors.

In [1]:
import os 
import sys
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd
from scipy.linalg import pinv, eigh
from scipy.stats import iqr
from scipy.sparse.linalg import eigsh
from scipy.stats import scoreatpercentile

# 2. Get current directory and split into parts
current_dir = os.getcwd()
parts = current_dir.split(os.sep)  # os.sep is the file separator ('/' or '\')

# 3. Reconstruct parent path (excluding last part)
parent_path = os.sep.join(parts[:-1])  # FORSE INUTILE

# 4. Add parent directory and its subfolders to Python path
sys.path.append(parent_path)

for root, dirs, files in os.walk(parent_path):
    sys.path.append(root)

sep = os.sep 

def main():
    
    sheet, IDf, trans, idt, method, q, algo = settings()

    if sheet == 'all':
       country = ['AT', 'BE', 'DE', 'EA', 'EL', 'ES', 'FR', 'IE', 'IT', 'NL', 'PT']
    else:
       country = [sheet]

     # Create output directory
    saveto = os.path.join(current_dir, f"data_TR{idt}")
    os.makedirs(saveto, exist_ok=True)

    for cc in range(len(country)):
        # ====================== #
        #   STEP 1: Load data    #
        # ====================== #
        sheet = country[cc]
        
        # Read Excel files using pandas
        try:
            # Load dataset (preserving column names exactly as in Excel)
            dataset = pd.read_excel(f"{sheet}data.xlsx", sheet_name='data')
            
            # Load info and convert to dictionary 
            info = pd.read_excel(f"{sheet}data.xlsx", sheet_name='info')
            spec = {col: info[col].values for col in info.columns}

            # Select transformations based on idt
            spec['TR'] = spec[f'TR{idt}']

            print(f"Successfully loaded data for {sheet}")
            
        except Exception as e:
            print(f"Error loading data for {sheet}: {str(e)}")
            continue  # Skip to next country if error occurs

        data = dataset.iloc[:, 1:].to_numpy(dtype=np.float64)  # Force float type
        titles = [str(x) for x in dataset.columns[1:]]  # Ensure strings
        dates = dataset.iloc[:, 0].to_numpy()  # Preserve original dtype
        
        # Optional date parsing if first column contains dates
        try:
            dates = pd.to_datetime(dates).to_numpy()  # Convert to datetime64
        except:
            pass  # Keep as-is if not date-like

        # ======================== #
        #   STEP 2: Set frequency  #
        # ======================== #
        if IDf == 'QM':
            # Case 1: quarterly aggregation (monthly to quarterly)
            data, dates, _ = aggregate(data, dates, spec) 
            dates = pd.to_datetime(dates).to_period('Q').to_timestamp().to_numpy()
        elif IDf == 'Q':
            # Case 2: only quarterly data
            locQ = np.array([freq == 'Q' for freq in spec['Frequency']])
            data = data[:, locQ]
            # Convert titles to numpy array for boolean indexing
            titles = np.array(titles)[locQ].tolist() if isinstance(titles, list) else titles[locQ]
        
           
            
            # Filter spec dictionary for quarterly series
            for key in ['TR', 'Class', 'Name', 'Aggregation', 'Frequency']:
                if key in spec:
                    spec[key] = [val for val, freq in zip(spec[key], spec['Frequency']) if freq == 'Q']
            
            data, dates, _ = aggregate(data, dates, spec)
            dates = pd.to_datetime(dates).to_period('Q').to_timestamp().to_numpy()

        elif IDf == 'M':
            # Case 3: only monthly data
            locM = np.array([freq == 'M' for freq in spec['Frequency']])
            data = data[:, locM]
            
            # Handle titles whether it's list or numpy array
            titles = np.array(titles)[locM].tolist() if isinstance(titles, list) else titles[locM]
            
            # Filter spec dictionary
            for key in ['TR', 'Class', 'Name', 'Aggregation', 'Frequency']:
                if key in spec:
                    spec[key] = [val for val, freq in zip(spec[key], spec['Frequency']) if freq == 'M']
        else:
            spec['agg'] = 0

        # ======================== #
        #  STEP 3: Transform Data  #
        # ======================== #
        spec['trans'] = trans
        xt = EA_transform(data, spec)

        # ===================== #
        #  STEP 4: Impute Data  #
        # ===================== #
        opts = {'maxiter': 1000, 'thresh': 1e-5, 'algo': algo}  # Set EM algorithm parameters

        # Convert dates to datetime if they're strings (assuming dates is a list/array)
        dates = pd.to_datetime(dates)  # Convert to datetime objects if needed

        # Find specific dates
        T19 = np.where(dates == pd.to_datetime('2019-10-01'))[0][0]  # Returns first index of Oct 1 2019
        T21 = np.where(dates == pd.to_datetime('2021-10-01'))[0][0]  # Returns first index of Oct 1 2021
        opts['T19'] = T19

        if method == -1:
            # Case -1: no imputation
            Xc = xt
            IDi = 'NA_'
            
        elif method == 0:
            # Case 0: impute only ragged edges
            opts['out'] = 0
            Xc = xt.copy()
            Xc, _ = EMimputation(xt, q, opts)
            IDi = ''
            
        elif method == 1:
            # Case 1: standard outlier imputation
            opts['out'] = 1
            Xc, pc = EMimputation(xt, q, opts)
            IDi = 'OA_'
            
        elif method == 2:
            # Case 2: impute real variables during Covid
            loc = [i for i, cls in enumerate(spec['Class']) if cls == 'R']
            Xnan = xt.copy()
            Xnan[T19+1:T21, loc] = np.nan
            Xc, pc = EMimputation(Xnan, q, opts)
            IDi = 'COV_'
            
        # Create and save timetable
        TT = pd.DataFrame(Xc, index=dates, columns=spec['Name'])
        TT.index.name = 'Date'
        
        # Define file name
        sIDi = f"_{IDi}" if IDi else IDi
        nsave = os.path.join(saveto, f"{sheet}data{IDf}{sIDi}_TR{idt}.xlsx")        
        
        # Save to Excel
        TT.to_excel(nsave, sheet_name=sheet)
        
        print(f"Processed and saved {sheet}")

    return Xc



def settings():
    """
    Command-line interface for model settings.
    
    Returns:
        sheet (str): Country identifier ('all' or specific)
        IDf (str): Frequency ('M', 'Q', or 'QM')
        trans (str): Transformation method ('light' or 'heavy')
        method (int/str): Imputation method (number or empty string)
        q (int): Number of factors for imputation
    """
    # Ask for default options
    use_default = input("Use default options? (y/n): ").strip().lower()
    
    if use_default == 'y':
        sheet = 'all'
        IDf = 'QM'
        trans = 'light'
        method = 0
        q = 99
        algo = 'SW'
    else:
        # Country selection
        sheet = input("Select country [default: all]: ").strip()
        if not sheet:
            print("No country specified, downloading data for all countries")
            sheet = 'all'
        
        # Frequency selection
        IDf = input("Select frequency (QM/Q/M/MF) [default: QM]: ").strip().upper()
        if IDf not in ('QM', 'Q', 'M', 'MF'):
            print("Invalid frequency, using <QM>")
            IDf = 'QM'
        
        # Transformation selection
        trans = input("Select transformation (light/heavy/BLT) [default: light]: ").strip().lower()
        if trans not in ('light', 'heavy', 'blt'):
            print("Invalid transformation, using <light>")
            trans = 'light'
        
        # Imputation selection
        imp = input("Impute missing values/outliers/Covid period? (y/n) [default: y]: ").strip().lower()
        if imp != 'n':
            method = input("Imputation method [default: 0]: ").strip()
            method = int(method) if method and method.isdigit() else 0
            
            q = input("Number of factors [default: 99]: ").strip()
            q = int(q) if q and q.isdigit() else 99
            
            # Algorithm selection
            algo = input("Select imputation algorithm (SW/BM) [default: SW]: ").strip().upper()
            if algo not in ('SW', 'BM'):
                print("Invalid algorithm, using 'SW'")
                algo = 'SW'
        else:
            method = -1
            q = 99
            algo = ''
    
    # Set transformation identifier
    if trans == 'light':
        idt = 2
    elif trans == 'heavy':
        idt = 1
    elif trans == 'blt':
        idt = 3
    
    # Mixed-frequency imputation warning
    if IDf == 'MF' and method > -1:
        print("Warning: Imputation of missing values not available for mixed-frequency data, setting <method> to -1")
        method = -1
    
    # Print summary
    print(f"Country: {sheet}; Frequency: {IDf}; trans: {trans}; method: {method}; q: {q}; algorithm: {algo}.")
    
    return sheet, IDf, trans, idt, method, q, algo

################################################################################

def aggregate(data, dates, spec):
    """
    Aggregates mixed-frequency (monthly-quarterly) data to quarterly frequency
    
    Parameters:
    -----------
    data : ndarray
        Input data (TxN) with NaNs
    dates : array-like
        Datetime objects (Tx1)
    spec : dict
        Contains:
        - Frequency: List of 'Q'/'M' for each variable
        - Aggregation: List of 1 (mean) or 2 (sum) for each variable
        - Name: Optional list of variable names
    
    Returns:
    --------
    dataQ : ndarray
        Quarterly aggregated data
    datesQ : ndarray
        Quarterly dates
    datasetQ : DataFrame
        DataFrame with dates as index
    """
    
    # Input validation
    n_vars = data.shape[1]

    if len(spec['Frequency']) != n_vars or len(spec['Aggregation']) != n_vars:
        raise ValueError('Identifiers must match data dimension')
    
    # Convert to DataFrame
    dates = pd.to_datetime(dates.flatten())
    temp = data.copy()
    tm = dates[len(dates)//2]
    
    # Process monthly variables
    for i in [i for i, freq in enumerate(spec['Frequency']) if freq == 'M']:
        idxNaN = np.isnan(data[:, i])
        valid_dates = dates[~idxNaN]
        
        if len(valid_dates) == 0:
            continue
            
        t0, t1 = min(valid_dates), max(valid_dates)
        
        # Handle single observation case
        if t0 == t1:
            if t0 < tm:
                t1 = None
            else:
                t0 = None
        
        # Check start of quarter
        if t0 is not None:
            tp = t0 + MonthEnd(1)
            tp1 = tp + MonthEnd(1)
            if (tp.quarter != t0.quarter) or (tp1.quarter != t0.quarter):
                temp[dates == t0, i] = np.nan
        
        # Check end of quarter
        if t1 is not None:
            tp = t1 - MonthEnd(1)
            tp1 = tp - MonthEnd(1)
            if (tp.quarter != t1.quarter) or (tp1.quarter != t1.quarter):
                temp[dates == t1, i] = np.nan
    
    # Create DataFrame
    var_names = spec.get('Name', [str(i) for i in range(n_vars)])
    df = pd.DataFrame(temp, index=dates, columns=var_names)
    
    # Identify flow variables (sum aggregation)
    if 'M' in spec['Frequency']:
        flow_vars = [var_names[i] for i, (freq, agg) in enumerate(zip(spec['Frequency'], spec['Aggregation'])) 
                     if freq == 'M' and agg == 2]
    else:
        flow_vars = []
    
    # Resample to quarterly
    df_q = df.resample('QS').mean()  # Default is mean (stock variables)
    # Handle flow variables (sum aggregation)
    for var in flow_vars:
        df_q[var] = df[var].resample('QS').sum()

    # 2. Shift labels to middle month (+2 months)
    df_q.index = df_q.index + pd.DateOffset(months=2)  # Move to 2nd month of quarter
    
    # Prepare outputs
    dataQ = df_q.to_numpy()
    datesQ = df_q.index.to_numpy()
    datasetQ = df_q.reset_index()
    
    return dataQ, datesQ, datasetQ

###############################################################################

def mdiff(X, s=1, k=1):
    """
    Performs k-th differences with a mixed-frequency panel of data
    
    Parameters:
    -----------
    X : ndarray
        Matrix of data with NaNs (TxN)
    s : int
        Period length (2 for quarters, 11 for years)
    k : int
        Order of differencing (default: 1)
    
    Returns:
    --------
    Y : ndarray
        Matrix of differenced data (TxN)
    """
    
    Y = np.full_like(X, np.nan)  # Initialize output with NaNs
    
    # Handle 1st order differences
    dX = X[s:,:] - X[:-s,:]  # 1st differences between periods s
    
    # Handle higher-order differences
    for _ in range(k-1):
        dX = dX[s:,:] - dX[:-s,:]
    
    # Fill output matrix
    Y[s*k:,:] = dX
    
    return Y

import numpy as np

###############################################################################

def remove_outliers(X, c=10):
    """
    Identifies and replaces outliers with NaNs based on the median and interquartile range.
    
    Parameters:
    -----------
    X : ndarray
        Input data matrix (TxN)
    c : float
        Threshold multiplier for outlier detection (default=10)
    
    Returns:
    --------
    X_cleaned : ndarray
        Data with outliers replaced by NaNs
    out : ndarray (bool)
        Boolean mask of outlier positions
    n : ndarray
        Number of outliers per column
    """
    # Calculate median and IQR for each column (ignoring NaNs)
    mX = np.nanmedian(X, axis=0)
    q75, q25 = np.nanpercentile(X, [75, 25], axis=0, method='linear')
    iqrX = q75 - q25 # most similar to matlab

    # Identify outliers (abs(X - median) > c * IQR)
    out = np.abs(X - mX) > c * iqrX
    
    # Replace outliers with NaN
    X_clean = X.copy()
    X_clean[out] = np.nan

    # Control for series with many zeros (>20% outliers)
    out_ratio = np.sum(out, axis=0) / X.shape[0]
    loc = out_ratio > 0.2
    out[:, loc] = False
    
    # Count outliers per column
    n = np.sum(out, axis=0)
    
    return X_clean, out, n

###############################################################################

def EA_transform(X, spec, c=1):
    """
    Applies transformations to variables based on spec['TR'] codes
    
    Parameters:
    -----------
    X : ndarray (TxN)
        Input data matrix
    spec : dict
        - TR: Transformation codes (1-6) for each variable
        - trans: 'light' or 'heavy' (for fractional TR handling)
        - Frequency: List of 'Q'/'M' for each variable
        - agg: 1 (balanced) or 0 (mixed frequencies)
    c : float
        Scaling factor (default=1)
    
    Returns:
    --------
    Xt : ndarray (TxN)
        Transformed data
    """
    # Initialize
    spec.setdefault('agg', 1)
    X = np.array(X, dtype=float)
    Xt = np.full_like(X, np.nan)
  
    # Validate
    if len(spec['TR']) != X.shape[1]:
        if len(spec['TR']) == X.shape[0]: 
            X = X.T
        else:
            raise ValueError("TR length must match X columns")
    
    # Check for negative values where logs are needed
    log_trs = [1, 2, 3]
    neg_mask = (X < 0) & np.isin(spec['TR'], log_trs)
    
    if np.any(neg_mask):
        bad_cols = np.where(np.any(neg_mask, axis=0))[0]
        print(f"Warning: Variables in positions {bad_cols} contain negative values")
        print("Logs are not admitted. Transforming without logs")
        
        # Replace log transformations with non-log equivalents
        for col in bad_cols:
            if spec['TR'][col] == 1:
                spec['TR'][col] = 4
            elif spec['TR'][col] == 2:
                spec['TR'][col] = 5
            elif spec['TR'][col] == 3:
                spec['TR'][col] = 6
    
    TR = spec['TR']
    # Apply transformations
   # Apply transformations
    if spec['agg'] == 1:  # Balanced panel
        masks = {
            1: TR == 1,  # c*log(x)
            2: TR == 2,  # Δlog(c*x)
            3: TR == 3,  # ΔΔlog(c*x)
            4: TR == 4,  # x (no transform)
            5: TR == 5,  # Δx
            6: TR == 6   # ΔΔx
        }
        
        Xt[:, masks[1]] = c * np.log(X[:, masks[1]])
        Xt[1:, masks[2]] = np.diff(c * np.log(X[:, masks[2]]), axis=0)
        Xt[2:, masks[3]] = np.diff(c * np.log(X[:, masks[3]]), n=2, axis=0)
        Xt[:, masks[4]] = X[:, masks[4]]
        Xt[1:, masks[5]] = np.diff(X[:, masks[5]], axis=0)
        Xt[2:, masks[6]] = np.diff(X[:, masks[6]], n=2, axis=0)
        
    else:  # Mixed frequencies
        is_m = np.array(spec['Frequency']) == 'M'
        is_q = ~is_m
        
        # TR=1 (logs)
        mask = TR == 1
        Xt[:, mask] = c * np.log(X[:, mask])
        
        # TR=2 (Δlog)
        Xt[1:, is_m & (TR == 2)] = np.diff(c * np.log(X[:, is_m & (TR == 2)]), axis=0)
        Xt[:, is_q & (TR == 2)] = mdiff(c * np.log(X[:, is_q & (TR == 2)]), 3, 1)
        
        # TR=3 (ΔΔlog)
        Xt[2:, is_m & (TR == 3)] = np.diff(c * np.log(X[:, is_m & (TR == 3)]), n=2, axis=0)
        Xt[:, is_q & (TR == 3)] = mdiff(c * np.log(X[:, is_q & (TR == 3)]), 3, 2)
        
        # TR=4 (no transform)
        Xt[:, TR == 4] = X[:, TR == 4]
        
        # TR=5 (Δx)
        Xt[1:, is_m & (TR == 5)] = np.diff(X[:, is_m & (TR == 5)], axis=0)
        Xt[:, is_q & (TR == 5)] = mdiff(X[:, is_q & (TR == 5)], 3, 1)
        
        # TR=6 (ΔΔx)
        Xt[2:, is_m & (TR == 6)] = np.diff(X[:, is_m & (TR == 6)], n=2, axis=0)
        Xt[:, is_q & (TR == 6)] = mdiff(X[:, is_q & (TR == 6)], 3, 2)
    
    return Xt

###############################################################################

def EMimputation(X, q0, opts=None):
    """
    Python implementation of EM imputation using factor models
    
    Parameters:
    -----------
    X : ndarray
        Input data matrix (TxN) with missing values as NaN
    q0 : int
        Number of factors (99 for Bai & Ng IC selection)
    opts : dict, optional
        Options dictionary with:
        - maxiter: Maximum iterations (default 500)
        - thresh: Convergence threshold (default 1e-6)
        - out: Outlier removal flag (default 0)
    
    Returns:
    --------
    X : ndarray
        Data with imputed values
    pc : dict
        Factor model results containing:
        - F: Factors (Txq)
        - C: Loadings (Nxq)
        - chi: Common component (TxN)
        - R: Idiosyncratic variances (N,)
        - d: Eigenvalues (q,)
    """
    # Set default options
    if opts is None:
        opts = {}
    opts.setdefault('maxiter', 1000)
    opts.setdefault('thresh', 1e-5)
    opts.setdefault('out', 0)

    if opts is None:   
        print(f"Maximum number of iterations set to {opts['maxiter']}")
        print(f"Threshold for EM objective set to {opts['thresh']:10f}")
    
    # Outlier removal
    if opts['out'] != 0:
        X, _, _ = remove_outliers(X)
    
    # Initial imputation with means 
    indNaN = np.isnan(X)
    mx = np.tile(np.nanmean(X, axis=0), (X.shape[0], 1))  # data mean, no NaNs
    sx = np.tile(np.nanstd(X, axis=0), (X.shape[0], 1))   # data std, no NaNs
    X[indNaN] = mx[indNaN]
    
    # Standardize
    mx = np.tile(np.mean(X, axis=0), (X.shape[0], 1))
    sx = np.tile(np.std(X, axis=0, ddof=1), (X.shape[0], 1))
    Xs = (X - mx) / sx
    
    # Determine number of factors
    q = BaiNg(Xs, 15, 2) if q0 == 99 else q0
    
    # Initial PCA
    pc = princfact(Xs, q)
    chi0 = pc['chi']
    
    EM = {}
    EM['chi'] = pc['chi']
    
    # EM algorithm
    err = 999
    j = 0
    
    while err > opts['thresh'] and j < opts['maxiter']:
        if j % 10 == 0:
            print(f'Running iteration {j}: error is {err:10f}')
        
        # Imputation step
        X[indNaN] = EM['chi'][indNaN] * sx[indNaN] + mx[indNaN]
        
        # Re-standardize
        mx = np.tile(np.mean(X, axis=0), (X.shape[0], 1))
        sx = np.tile(np.std(X, axis=0, ddof=1), (X.shape[0], 1))
        Xs = (X - mx) / sx
        
        # Update number of factors if needed
        q = BaiNg(Xs, 20, 2) if q0 == 99 else q0
        
        # PCA step
        EM = princfact(Xs, q)
        
        # Compute convergence criterion
        dchi = EM['chi'] - chi0
        err = np.sum(dchi**2) / np.sum(chi0**2)
        
        # Update for next iteration
        chi0 = EM['chi']
        j += 1
        
        if err < opts['thresh'] and j < opts['maxiter']:
            print(f'EM converged after {j} iterations')
    
    return X, EM

###############################################################################
def princfact(X, q, method=2, st=1):
    """
    Estimates factors via principal components with NaN handling
    
    Parameters:
    -----------
    X : ndarray
        Input data matrix (TxN)
    q : int
        Number of factors to extract
    method : int
        Normalization method (0-3, default=2)
    st : int
        Standardization flag (0/1, default=0)
    
    Returns:
    --------
    pc : dict
        Contains:
        - F: Factors (Txq)
        - C: Loadings (Nxq)
        - chi: Common component (TxN)
        - R: Idiosyncratic variances (N,)
        - d: Eigenvalues (q,)
    """
    T, N = X.shape
    
    # Handle NaNs
    if np.isnan(X).any():
        X = np.where(np.isnan(X), np.nanmean(X), X)
        print('Detected NaNs in original data matrix: replaced with unconditional mean of non-NaNs values')
    
    # Standardize if requested
    if st == 1:
        X = (X - np.mean(X, axis=0)) / np.std(X, axis=0, ddof=1)
    
    # Validate method
    if method not in [0, 1, 2, 3]:
        print('The value for <method> is outside the bounds, setting <method> to 2')
        method = 2
    
    # Compute covariance matrix
    cov_mat = np.cov(X.T, bias = True)
    
    # Factor extraction
    if method != 3:
        # Standard NxN covariance approach
        d, v = eigsh(cov_mat, k = q, which = 'LM')  # Get q largest eigenvalues/vectors
        d = np.flip(d)
        v = np.flip(v, axis=1)
        
        # Sign flip for consistency
        if np.sum(v) < 0:
            v = -v
            
        if method == 0:
            f = X @ v
            C = v
        elif method == 1:
            f = X @ v @ np.diag(1/np.sqrt(d))
            C = v @ np.diag(np.sqrt(d))
        elif method == 2:
            f = X @ v / np.sqrt(N)
            C = v * np.sqrt(N)
    else:
        # TxT covariance approach (method 3)
        cov_mat_t = np.cov(X, bias = True)
        d, v = eigsh(cov_mat_t, k = q, which='LM')  # Get q largest eigenvalues/vectors
        d = np.flip(d)
        v = np.flip(v, axis=1)
        
        if np.sum(v) < 0:
            v = -v
            
        f = v * np.sqrt(T)
        C = (f.T @ X).T / T
    
    # Common component and residuals
    chi = f @ C.T
    R = np.diag(np.diag(np.cov(X - chi, rowvar=False)))
    
    return {
        'F': f,
        'C': C,
        'chi': chi,
        'R': R,
        'd': d
    }

###############################################################################
def BaiNg(X, qmax=None, method=2):
    """
    Selects number of factors using Bai & Ng (2002) information criteria
    
    Parameters:
    -----------
    X : ndarray
        Input data matrix (TxN)
    qmax : int
        Maximum number of factors to test (default=sqrt(N))
    method : int
        Criterion method (1-3, default=2)
    
    Returns:
    --------
    q : int
        Number of factors selected
    """
    T, N = X.shape
    
    # Set defaults
    if qmax is None:
        qmax = int(np.sqrt(N))
    
    # Handle NaNs
    if np.isnan(X).any():
        X = np.where(np.isnan(X), np.nanmean(X), X)
        print('Detected NaNs in original data matrix: replaced with unconditional mean of non-NaNs values')
    
    # Standardize data
    X = (X - np.ones((T, 1)) @ (np.sum(X, axis=0).reshape(1, -1) / T))/ np.std(X, axis=0, ddof=1)
    
    # Initialize criterion
    CT = np.zeros(qmax)
    
    # Select penalty
    if method == 1:
        CT = np.log(N*T/(N+T)) * np.arange(1, qmax+1) * (N+T)/(N*T)
    elif method == 2:
        CT = ((N+T)/(N*T)) * np.log(min(N, T)) * np.arange(1, qmax+1)
    elif method == 3:
        CT = np.arange(1, qmax+1) * np.log(min(N, T))/min(N, T)
    else:
        raise ValueError("Method must be 1, 2, or 3")
    
    # Principal components
    if T < N:
        # Use TxT covariance matrix
        cov_mat = np.cov(X)
        d, v = eigsh(cov_mat, k=qmax, which='LM')  # Get qmax largest eigenvalues/vectors
        d = np.flip(d)
        v = np.flip(v, axis=1)
        Fhat = np.sqrt(T) * v
        Lhat = X.T @ Fhat / T
    else:
        # Use NxN covariance matrix
        cov_mat = np.cov(X.T, bias = True)
        d, v = eigsh(cov_mat,k=qmax, which='LM')  # Get qmax largest eigenvalues/vectors
        d = np.flip(d)
        v = np.flip(v, axis=1)
        Lhat = np.sqrt(N) * v
        Fhat = X @ Lhat / N

    # Information criterion
    Sigma = np.zeros(qmax+1)
    IC1 = np.zeros(qmax+1)
    
    for qq in range(qmax, 0, -1):
        chat = Fhat[:, :qq] @ Lhat[:, :qq].T
        ehat = X - chat
        Sigma[qq-1] = np.mean(np.sum(ehat * ehat, axis=0) / T)
        IC1[qq-1] = np.log(Sigma[qq-1]) + CT[qq-1]
    
    # Case with no factors
    Sigma[qmax] = np.mean(np.sum(X * X, axis=0) / T)
    IC1[qmax] = np.log(Sigma[qmax])

    # Select number of factors
    q = np.argmin(IC1)+1
    if q > qmax:
        q = 0

    return q

###############################################################################
def kalman(Y, pars):
    """
    Python implementation of MATLAB's Kalman Filter and Smoother
    Maintains identical input/output structure and variable names
    """
    # Unpack parameters
    A = pars['A']
    C = pars['C']
    R = pars['R']
    Q = pars['Q']
    
    # Initialize parameters
    Z0 = pars.get('Z00', np.zeros((C.shape[1], 1)))
    P0 = pars.get('P00', 10 * np.eye(C.shape[1]))
    mu = pars.get('mu', np.zeros((C.shape[1], 1)))
    beta = pars.get('beta', np.zeros((1, 2)))
    
    ns = C.shape[1]  # Number of states
    Y = Y.T  
    N, T = Y.shape
    
    # Initialize output structure 
    KF = {
        'Zttm': np.nan * np.zeros((ns, T)),
        'Pttm': np.nan * np.zeros((ns, ns, T)),
        'Ztt': np.nan * np.zeros((ns, T+1)),
        'Ptt': np.nan * np.zeros((ns, ns, T+1)),
        'Kt': np.zeros((ns, N, T)),
        'vt': np.zeros((N, T)),
        'loglik': 0,
        'Z00': Z0,
        'P00': P0
    }
    
    KF['Ztt'][:, [0]] = Z0
    KF['Ptt'][:, :, 0] = P0
    mu = np.hstack([np.zeros((ns, 1)), np.tile(mu, (1, T-1))])
    
    # Kalman Filter %%
    for t in range(T):
        # Prediction Step
        KF['Zttm'][:, t] = mu[:, t] + A @ KF['Ztt'][:, t]
        KF['Pttm'][:, :, t] = A @ KF['Ptt'][:, :, t] @ A.T + Q
        KF['Pttm'][:, :, t] = 0.5 * (KF['Pttm'][:, :, t] + KF['Pttm'][:, :, t].T)
        
        # Update Step
        idx = ~np.isnan(Y[:, t])
        yt = Y[idx, t]
        Ct = C[idx, :]
        Rt = R[np.ix_(idx, idx)]
        bt = beta[0, idx] if np.any(beta) else beta
        
        if len(yt) == 0:
            KF['Ztt'][:, t+1] = KF['Zttm'][:, t]
            KF['Ptt'][:, :, t+1] = KF['Pttm'][:, :, t]
        else:
            Kt_part = KF['Pttm'][:, :, t] @ Ct.T @ np.linalg.pinv(Ct @ KF['Pttm'][:, :, t] @ Ct.T + Rt)
            KF['Kt'][:, idx, t] = Kt_part
            KF['vt'][idx, t] = yt - Ct @ KF['Zttm'][:, [t]] - bt @ np.array([[1], [t+1]])
            
            KF['Ztt'][:, t+1] = KF['Zttm'][:, t] + KF['Kt'][:, :, t] @ KF['vt'][:, [t]]
            KF['Ptt'][:, :, t+1] = KF['Pttm'][:, :, t] - Kt_part @ Ct @ KF['Pttm'][:, :, t]
            KF['Ptt'][:, :, t+1] = 0.5 * (KF['Ptt'][:, :, t+1] + KF['Ptt'][:, :, t+1].T)
            
            # Log-likelihood calculation
            innov_cov = Ct @ KF['Pttm'][:, :, t] @ Ct.T + Rt
            KF['loglik'] += 0.5 * (np.log(np.linalg.det(np.linalg.pinv(innov_cov))) - 
                                  KF['vt'][idx, t].T @ np.linalg.pinv(innov_cov) @ KF['vt'][idx, t])
    
    #  Kalman Smoother %%
    KF['ZtT'] = np.zeros((ns, T))
    KF['PtT'] = np.zeros((ns, ns, T))
    KF['PtTm'] = np.zeros((ns, ns, T))
    
    KF['ZtT'][:, T-1] = KF['Ztt'][:, T]
    KF['PtT'][:, :, T-1] = KF['Ptt'][:, :, T]
    KF['PtTm'][:, :, T-1] = (np.eye(ns) - KF['Kt'][:, :, T-1] @ C) @ A @ KF['Ptt'][:, :, T-1]
    
    Pts = np.zeros((ns, ns, T))
    Pts[:, :, T-1] = KF['Ptt'][:, :, T-1] @ A.T @ pinv(KF['Pttm'][:, :, T-1])
    
    for t in range(T-1, 0, -1):
        KF['ZtT'][:, t-1] = (KF['Ztt'][:, t] + 
                             Pts[:, :, t] @ (KF['ZtT'][:, t] - KF['Zttm'][:, t]))
        
        KF['PtT'][:, :, t-1] = (KF['Ptt'][:, :, t] + 
                                Pts[:, :, t] @ (KF['PtT'][:, :, t] - KF['Pttm'][:, :, t]) @ Pts[:, :, t].T)
        
        if t > 1:
            Pts[:, :, t-1] = KF['Ptt'][:, :, t-1] @ A.T @ pinv(KF['Pttm'][:, :, t-1])
            KF['PtTm'][:, :, t-1] = (KF['Ptt'][:, :, t] @ Pts[:, :, t-1].T + 
                                     Pts[:, :, t] @ (KF['PtTm'][:, :, t] - A @ KF['Ptt'][:, :, t]) @ Pts[:, :, t-1].T)
    
    KF['ZtT'] = KF['ZtT'].T  
    
    return KF

###################################################################################


if __name__ == '__main__':
    Xc = main()


Country: EA; Frequency: QM; trans: light; method: 0; q: 99; algorithm: SW.
Error loading data for EA: [Errno 2] No such file or directory: 'EAdata.xlsx'


UnboundLocalError: cannot access local variable 'Xc' where it is not associated with a value